# De Novo Peptide Sequencing Database Explorer

This notebook inspects the contents of `denovo.db` and creates visualizations of the relationships between authors, algorithms, publications, and affiliations in the field of de novo peptide sequencing.

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import networkx as nx
from collections import Counter

sns.set_theme(style="whitegrid", palette="muted")
conn = sqlite3.connect("denovo.db")

## 1. Database Overview

In [2]:
tables = ["algorithm", "publication", "author", "affiliation", "country", "city",
          "author_affiliation", "publication_author", "publication_algorithm"]

for t in tables:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn).iloc[0, 0]
    print(f"{t:30s} {n:>5d} rows")

algorithm                         58 rows
publication                       63 rows
author                           262 rows
affiliation                      119 rows
country                           16 rows
city                              55 rows
author_affiliation               354 rows
publication_author               421 rows
publication_algorithm             64 rows


## 2. All Algorithms

In [3]:
df_alg = pd.read_sql("SELECT name, repository FROM algorithm ORDER BY name", conn)
df_alg

,name,repository
0,AdaNovo,NaN
1,BiATNovo,NaN
2,Casanovo,https://github.com/Noble-Lab/casanovo
3,Casanovo-DB,NaN
4,Cascadia,NaN
5,ContraNovo,https://github.com/BEAM-Labs/ContraNovo
6,CrossNovo,https://github.com/BEAM-Labs/denovo
7,CycloNovo,https://github.com/bbehsaz/cyclonovo
8,DIANovo,NaN
9,DePS,NaN


## 3. Publications

## 4. Publications Timeline

In [4]:
df_timeline = df_pub.dropna(subset=["publication_date"]).copy()
df_timeline["year"] = df_timeline["publication_date"].dt.year
year_counts = df_timeline["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(year_counts.index, year_counts.values, color=sns.color_palette()[0], edgecolor="white")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Publications")
ax.set_title("De Novo Peptide Sequencing Publications per Year")
ax.set_xticks(year_counts.index)
plt.tight_layout()
plt.savefig("plots/publications_per_year.png", dpi=150, bbox_inches="tight")
plt.show()

NameError: name 'df_pub' is not defined

## 5. Publication Types

In [ ]:
type_counts = df_pub["publication_type"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(type_counts.values, labels=type_counts.index, autopct="%1.0f%%",
       colors=sns.color_palette("pastel"))
ax.set_title("Publication Types")
plt.tight_layout()
plt.savefig("plots/publication_types.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Top Authors by Number of Publications

In [ ]:
df_top_authors = pd.read_sql("""
    SELECT a.name, COUNT(pa.publication_id) as num_publications
    FROM author a
    JOIN publication_author pa ON a.id = pa.author_id
    GROUP BY a.id
    ORDER BY num_publications DESC
    LIMIT 20
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_top_authors["name"][::-1], df_top_authors["num_publications"][::-1],
        color=sns.color_palette("viridis", len(df_top_authors)))
ax.set_xlabel("Number of Publications")
ax.set_title("Top 20 Authors by Publication Count")
plt.tight_layout()
plt.show()

## 7. Geographic Distribution of Affiliations

In [ ]:
df_geo = pd.read_sql("""
    SELECT c.name as country, COUNT(DISTINCT af.id) as num_affiliations,
           COUNT(DISTINCT aa.author_id) as num_authors
    FROM country c
    JOIN affiliation af ON af.country_id = c.id
    LEFT JOIN author_affiliation aa ON aa.affiliation_id = af.id
    GROUP BY c.id
    ORDER BY num_authors DESC
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(df_geo["country"][::-1], df_geo["num_affiliations"][::-1],
             color=sns.color_palette("coolwarm", len(df_geo)))
axes[0].set_xlabel("Number of Affiliations")
axes[0].set_title("Affiliations per Country")

axes[1].barh(df_geo["country"][::-1], df_geo["num_authors"][::-1],
             color=sns.color_palette("coolwarm", len(df_geo)))
axes[1].set_xlabel("Number of Authors")
axes[1].set_title("Authors per Country")

plt.tight_layout()
plt.show()

df_geo

## 8. Top Institutions

In [ ]:
df_inst = pd.read_sql("""
    SELECT af.name as institution, c.name as country,
           COUNT(DISTINCT aa.author_id) as num_authors
    FROM affiliation af
    JOIN country c ON af.country_id = c.id
    JOIN author_affiliation aa ON aa.affiliation_id = af.id
    GROUP BY af.name
    ORDER BY num_authors DESC
    LIMIT 15
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
labels = [f"{r['institution']} ({r['country']})" for _, r in df_inst.iterrows()]
ax.barh(labels[::-1], df_inst["num_authors"][::-1],
        color=sns.color_palette("rocket", len(df_inst)))
ax.set_xlabel("Number of Authors")
ax.set_title("Top 15 Institutions by Author Count")
plt.tight_layout()
plt.show()

## 9. Co-authorship Network

In [ ]:
df_coauth = pd.read_sql("""
    SELECT pa1.author_id as a1, pa2.author_id as a2, COUNT(*) as weight
    FROM publication_author pa1
    JOIN publication_author pa2 ON pa1.publication_id = pa2.publication_id
    WHERE pa1.author_id < pa2.author_id
    GROUP BY pa1.author_id, pa2.author_id
""", conn)

author_names = dict(pd.read_sql("SELECT id, name FROM author", conn).values)

G = nx.Graph()
for _, row in df_coauth.iterrows():
    n1, n2 = author_names[row["a1"]], author_names[row["a2"]]
    G.add_edge(n1, n2, weight=row["weight"])

# Filter to authors with >= 3 publications for readability
prolific = {r[0] for r in pd.read_sql("""
    SELECT a.name FROM author a
    JOIN publication_author pa ON a.id = pa.author_id
    GROUP BY a.id HAVING COUNT(*) >= 3
""", conn).values}

G_sub = G.subgraph([n for n in G.nodes if n in prolific]).copy()

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(G_sub, k=1.5, seed=42)
degrees = dict(G_sub.degree())
node_sizes = [degrees[n] * 80 for n in G_sub.nodes]
edges = G_sub.edges(data=True)
edge_weights = [e[2]["weight"] for e in edges]

nx.draw_networkx_edges(G_sub, pos, alpha=0.3, width=[w * 0.5 for w in edge_weights], ax=ax)
nx.draw_networkx_nodes(G_sub, pos, node_size=node_sizes,
                       node_color=[degrees[n] for n in G_sub.nodes],
                       cmap=plt.cm.YlOrRd, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G_sub, pos, font_size=7, ax=ax)
ax.set_title("Co-authorship Network (authors with ≥ 3 publications)")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Network: {G_sub.number_of_nodes()} authors, {G_sub.number_of_edges()} co-authorship links")

## 10. Algorithm Timeline

In [ ]:
df_alg_timeline = pd.read_sql("""
    SELECT a.name as algorithm, MIN(p.publication_date) as first_pub
    FROM algorithm a
    JOIN publication_algorithm pa ON a.id = pa.algorithm_id
    JOIN publication p ON pa.publication_id = p.id
    WHERE p.publication_date IS NOT NULL
    GROUP BY a.id
    ORDER BY first_pub
""", conn)
df_alg_timeline["first_pub"] = pd.to_datetime(df_alg_timeline["first_pub"])

fig, ax = plt.subplots(figsize=(14, 12))
y_pos = range(len(df_alg_timeline))
colors = sns.color_palette("husl", len(df_alg_timeline))

ax.barh(y_pos, [1]*len(df_alg_timeline), left=mdates.date2num(df_alg_timeline["first_pub"]),
        height=0.6, color=colors, alpha=0)

for i, (_, row) in enumerate(df_alg_timeline.iterrows()):
    ax.plot(row["first_pub"], i, "o", color=colors[i], markersize=8)
    ax.text(row["first_pub"], i, f"  {row['algorithm']}", va="center", fontsize=7)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.set_yticks([])
ax.set_title("Algorithm Publication Timeline (first preprint appearance)")
ax.set_xlabel("Date")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("plots/publication_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

## 10b. Innovations Timeline

A horizontal timeline of key innovations in deep-learning-based de novo peptide sequencing, color-coded by core architecture type.

In [ ]:
band_defs = [
    ("CNN + RNN",         "#4C72B0",  2.0),
    ("Transformer (AR)",  "#DD8452",  6.0),
    ("GNN",               "#937860",  1.2),
    ("CNN",               "#8172B3",  1.2),
    ("Transformer (NAR)", "#55A868",  1.6),
    ("Diffusion",         "#C44E52",  1.2),
]

BAND_GAP = 0.15

bands = {}
y_cursor = 0
for name, color, height in band_defs:
    bot = y_cursor
    top = y_cursor + height
    bands[name] = (bot, top, (bot + top) / 2, color, height)
    y_cursor = top + BAND_GAP

# Pull innovations from the database
df_innov = pd.read_sql("""
    SELECT a.name, a.algorithm_family, a.short_description,
           MIN(p.publication_date) AS first_pub
    FROM algorithm a
    JOIN publication_algorithm pa ON a.id = pa.algorithm_id
    JOIN publication p ON pa.publication_id = p.id
    WHERE a.algorithm_family IS NOT NULL
      AND p.publication_date IS NOT NULL
    GROUP BY a.id
    ORDER BY first_pub
""", conn)
df_innov["first_pub"] = pd.to_datetime(df_innov["first_pub"])

# Compute y_frac within each band to avoid label overlap.
# Items < 120 days apart in the same band cycle through tiers.
Y_TIERS = [0.5, 0.8, 0.2, 0.9, 0.1, 0.65, 0.35, 0.95, 0.05, 0.72, 0.28, 0.85, 0.15, 0.58, 0.42]
MIN_GAP_DAYS = 180

items = []
last_date_at_tier = {}  # (family, tier_idx) -> last date placed there
for _, row in df_innov.iterrows():
    fam = row["algorithm_family"]
    if fam not in bands:
        continue
    date = row["first_pub"]
    placed = False
    for tier_idx, y_frac in enumerate(Y_TIERS):
        key = (fam, tier_idx)
        if key not in last_date_at_tier or (date - last_date_at_tier[key]).days >= MIN_GAP_DAYS:
            last_date_at_tier[key] = date
            items.append((date, row["name"], row["short_description"], fam, y_frac))
            placed = True
            break
    if not placed:
        last_date_at_tier[(fam, 0)] = date
        items.append((date, row["name"], row["short_description"], fam, 0.5))

fig, ax = plt.subplots(figsize=(30, 15))

for bname, (bot, top, center, color, bh) in bands.items():
    ax.axhspan(bot, top, color=color, alpha=0.09, zorder=0)
    ax.axhline(bot, color=color, linewidth=0.5, alpha=0.25)
    ax.axhline(top, color=color, linewidth=0.5, alpha=0.25)

band_centers = [bands[n][2] for n, _, _ in band_defs]
ax.set_yticks(band_centers)
ax.set_yticklabels([n for n, _, _ in band_defs], fontsize=13, fontweight="bold")
for lbl, (_, color, _) in zip(ax.get_yticklabels(), band_defs):
    lbl.set_color(color)

for date, name, desc, arch, y_frac in items:
    bot, top, center, color, bh = bands[arch]
    y = bot + y_frac * bh

    ax.plot(date, y, "o", color=color, markersize=11, zorder=4,
            markeredgecolor="white", markeredgewidth=1.3)

    va = "bottom" if y_frac >= 0.5 else "top"
    y_off = 8 if y_frac >= 0.5 else -8

    ax.annotate(name, xy=(date, y), xytext=(0, y_off),
                textcoords="offset points",
                fontsize=9, fontweight="bold", va=va, ha="center",
                color=color, zorder=5)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.set_xlim(pd.Timestamp("2016-10-01"), pd.Timestamp("2026-12-01"))
total_h = sum(h for _, _, h in band_defs) + BAND_GAP * (len(band_defs) - 1)
ax.set_ylim(-0.15, total_h + 0.15)
ax.set_xlabel("Year", fontsize=14)
ax.tick_params(axis="x", labelsize=13)
ax.tick_params(axis="y", length=0)
ax.set_title("Key Innovations in Deep-Learning-Based De Novo Peptide Sequencing",
             fontsize=20, fontweight="bold", pad=20)
ax.yaxis.grid(False)
sns.despine(left=True)
plt.tight_layout()
plt.savefig("plots/innovations_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Journals / Venues

In [ ]:
df_journals = pd.read_sql("""
    SELECT journal, COUNT(*) as cnt
    FROM publication
    WHERE journal IS NOT NULL
    GROUP BY journal
    ORDER BY cnt DESC
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
top_journals = df_journals.head(15)
ax.barh(top_journals["journal"][::-1], top_journals["cnt"][::-1],
        color=sns.color_palette("mako", len(top_journals)))
ax.set_xlabel("Number of Publications")
ax.set_title("Top Publication Venues")
plt.tight_layout()
plt.show()

## 12. Custom Queries

Use the cell below to run custom SQL queries against the database.

In [ ]:
# Example: find all publications by a specific author
pd.read_sql("""
    SELECT p.title, p.publication_date, a2.name as algorithm
    FROM author a
    JOIN publication_author pa ON a.id = pa.author_id
    JOIN publication p ON pa.publication_id = p.id
    LEFT JOIN publication_algorithm pal ON p.id = pal.publication_id
    LEFT JOIN algorithm a2 ON pal.algorithm_id = a2.id
    WHERE a.name = 'Siqi Sun'
    ORDER BY p.publication_date DESC
""", conn)

In [ ]:
# Example: find all affiliations for an author
pd.read_sql("""
    SELECT a.name as author, af.name as institution, af.department, c.name as country
    FROM author a
    JOIN author_affiliation aa ON a.id = aa.author_id
    JOIN affiliation af ON aa.affiliation_id = af.id
    JOIN country c ON af.country_id = c.id
    WHERE a.name = 'Xiang Zhang'
""", conn)

In [ ]:
conn.close()